# Blocks

> Processing steps that compose: arrays in, arrays out, as in Sionna.

In [ ]:
#| default_exp block

In [ ]:
#| hide
from nbdev.showdoc import *

A `Block` is one step of a simulation, written like a [Sionna block](https://nvlabs.github.io/sionna/phy/api/block.html):
it takes arrays and returns arrays, its work is in `call`, and `build` sets up what depends on the inputs' shapes,
once, before the first call. A larger step is a block whose `call` runs smaller ones, or a `Pipeline` of them.

Unlike Sionna's, a comm-core block is not a `torch.nn.Module`: the arrays are numpy arrays or torch tensors, and
the block does not convert them. Two things are added for a network simulation:

* **Names.** A block's inputs are the parameters of its `call`, and its `outputs` name what it returns. A
  `Pipeline` passes each block what it needs by name. `Named` gives names to any callable, e.g. a Sionna block.
* **Episodes and rounds.** `reset(seed)` starts an episode: it reseeds the block's random generator (`rng`) and
  its sub-blocks, each with its own seed derived from `seed`. `update(time)` starts a round, e.g. for a fading
  that changes in time or a power control that learns.

The radio channels are made of blocks: a `Fading` gives `h`, a `PowerControl` the power, then the SINR, the
interference, a PHY abstraction or the bits, and the airtime (see `system`).

In [ ]:
#| export
from __future__ import annotations

import inspect
from typing import Any, Callable, Iterator, Mapping, Sequence

import numpy as np

In [ ]:
#| export
def _child_seed(seed: int | None, i: int) -> int | None:
    "The seed of the `i`-th part of something seeded with `seed` (None: unseeded)."
    return None if seed is None else int(np.random.SeedSequence([seed, i]).generate_state(1)[0])


def _shape(x):
    "The shapes of the inputs, nested like them (for `build`)."
    if hasattr(x, 'shape'):
        return tuple(x.shape)
    if isinstance(x, (list, tuple)):
        return type(x)(_shape(v) for v in x) if type(x) in (list, tuple) else ()
    if isinstance(x, dict):
        return {k: _shape(v) for k, v in x.items()}
    return ()


def _params(f) -> list:
    return [p for p in inspect.signature(f).parameters.values() if p.kind not in (p.VAR_POSITIONAL, p.VAR_KEYWORD)]

In [ ]:
#| export
class Block:
    "A processing step: arrays in, arrays out (numpy or torch). Subclasses write `call`, and `build` if they need the inputs' shapes."

    outputs: tuple = () # Names of what `call` returns, in order (for `Pipeline`)

    @property
    def built(self) -> bool:
        "Whether `build` has run."
        return getattr(self, '_built', False)

    @property
    def rng(self) -> np.random.Generator:
        "The block's random generator, seeded by `reset`."
        if getattr(self, '_rng', None) is None:
            self._rng = np.random.default_rng()
        return self._rng

    @rng.setter
    def rng(self, value: np.random.Generator) -> None:
        self._rng = value

    @property
    def inputs(self) -> tuple:
        "Names of the inputs: the parameters of `call`."
        return tuple(p.name for p in _params(self.call))

    @property
    def required(self) -> tuple:
        "The inputs without a default."
        return tuple(p.name for p in _params(self.call) if p.default is p.empty)

    def children(self) -> list:
        "The blocks inside this one: its attributes that are blocks, or lists, tuples or dicts of blocks."
        found, seen = [], set()
        for name, value in vars(self).items():
            if name == '_rng':
                continue
            if isinstance(value, dict):
                items = list(value.values())
            elif isinstance(value, (list, tuple)):
                items = value
            else:
                items = (value,)
            if not items or not isinstance(items[0], Block):   # a list of other things (e.g. positions): skipped
                continue
            for item in items:
                if isinstance(item, Block) and id(item) not in seen:
                    found.append(item)
                    seen.add(id(item))
        return found

    def walk(self) -> Iterator[Block]:
        "This block and every block inside it, each once."
        seen, todo = set(), [self]
        while todo:
            block = todo.pop(0)
            if id(block) not in seen:
                seen.add(id(block))
                yield block
                todo += block.children()

    def build(self, *shapes, **kwshapes) -> None:
        "Set up what depends on the shapes of the inputs (given like `call` gets the inputs); runs once, before the first `call`."

    def call(self, *args, **kwargs) -> Any:
        "The block's work: its inputs -> its outputs (one value, a tuple in the order of `outputs`, or a dict by name)."
        raise NotImplementedError(f"{type(self).__name__} has no `call`")

    def __call__(self, *args, **kwargs) -> Any:
        if not self.built:
            self.build(*_shape(args), **_shape(kwargs))
            self._built = True
        return self.call(*args, **kwargs)

    def reset(
        self,
        seed: int | None = None, # The episode's seed; None: keep the random generators as they are
    ) -> None:
        "A new episode: reseed this block, and its sub-blocks with seeds derived from `seed`."
        if seed is not None:
            self._rng = np.random.default_rng(_child_seed(seed, 0))
        for i, child in enumerate(self.children(), 1):
            child.reset(_child_seed(seed, i))

    def update(
        self,
        time: float, # When the round starts
    ) -> None:
        "A new round of transmissions (e.g. a fading changing in time, a power control learning); passed to the sub-blocks."
        for child in self.children():
            child.update(time)

In [ ]:
show_doc(Block.call)

In [ ]:
show_doc(Block.build)

In [ ]:
show_doc(Block.reset)

A block is a class with a `call`. Here a fading that draws `h` for each link, with the number of subcarriers
fixed by the first input it sees:

In [ ]:
class Draw(Block):
    outputs = ('h',)
    def __init__(self, num_subcarriers=None):
        self.num_subcarriers = num_subcarriers
    def build(self, sender):                    # the shape of `sender`, as `call` gets it
        if self.num_subcarriers is None:
            self.num_subcarriers = 1
    def call(self, sender):
        n = (len(sender), self.num_subcarriers)
        return (self.rng.standard_normal(n) + 1j * self.rng.standard_normal(n)) / np.sqrt(2)

d = Draw()
assert not d.built and d.inputs == ('sender',) and d.required == ('sender',)
h = d(np.array(['a', 'b', 'c']))
assert d.built and h.shape == (3, 1) and d.num_subcarriers == 1
# reset(seed): the same draws again
d.reset(seed=0); first = d(np.array(['a', 'b']))
d.reset(seed=0); assert (d(np.array(['a', 'b'])) == first).all()
d.reset(); assert not (d(np.array(['a', 'b'])) == first).all()     # no seed: the generator goes on

In [ ]:
# sub-blocks: found in attributes, lists and dicts, reset with their own seeds, updated with the block
class Pair(Block):
    def __init__(self):
        self.first, self.others = Draw(1), [Draw(1)]
        self.times = []
    def update(self, time):
        self.times.append(time)
        super().update(time)

p = Pair()
assert p.children() == [p.first, p.others[0]] and len(list(p.walk())) == 3
p.reset(seed=1)
x, y = p.first(['a']), p.others[0](['a'])
assert x != y                                                     # different seeds
p.reset(seed=1); assert p.first(['a']) == x
p.update(2.0); assert p.times == [2.0]

## Pipeline

A `Pipeline` runs blocks in a row. It keeps the named arrays of a call: its inputs, then each block's outputs. Each
block gets the ones its `call` takes, by name, and adds its outputs; a later block can replace an array (e.g.
`Interference` narrows `active`). It returns all of them, as a dict.

With `inputs` given, the wiring is checked when the pipeline is made: a block needing an array that is neither an
input nor an earlier block's output is an error then, not in the middle of a simulation. An input with a default
in a block's `call` is optional: the block gets it if it is there.

A pipeline is itself a block, so pipelines nest. When a list does not fit (branches, feedback), a block with its
own `call` can run its sub-blocks in any order, as in Sionna.

In [ ]:
#| export
def _name(block) -> str:
    return type(block).__name__


def _store(block, out, state: dict) -> None:
    "Add a block's outputs to the named arrays."
    if isinstance(out, Mapping):
        state.update(out)
        return
    names = tuple(block.outputs)
    if not names:
        return
    values = out if isinstance(out, tuple) and len(names) != 1 else (out,)
    if len(values) != len(names):
        raise ValueError(f"{_name(block)} returned {len(values)} values for its outputs {names}")
    state.update(zip(names, values))


class Pipeline(Block):
    "Blocks in a row, wired by name: each gets the inputs and earlier outputs its `call` takes, and adds its outputs."

    def __init__(
        self,
        blocks: Sequence[Block], # The blocks, in order
        inputs: Sequence[str] | None = None, # Names the pipeline is called with; given, the wiring is checked now
    ):
        self.blocks = list(blocks)
        for block in self.blocks:
            if not isinstance(block, Block):
                raise TypeError(f"{block!r} is not a Block (wrap a callable in `Named`)")
        available = set(inputs or ())
        required, optional, outputs = [], [], []
        for block in self.blocks:
            missing = [n for n in block.required if n not in available]
            if inputs is not None and missing:
                raise ValueError(f"{_name(block)} needs {missing}: not an input, and no block before it gives it")
            required += [n for n in missing if n not in required]
            optional += [n for n in block.inputs if n not in available and n not in required and n not in optional]
            outputs += [n for n in block.outputs if n not in outputs]
            available |= set(block.outputs)
        self._required = tuple(inputs) if inputs is not None else tuple(required)
        self._inputs = self._required + tuple(n for n in optional if n not in self._required)
        self.outputs = tuple(outputs)

    @property
    def inputs(self) -> tuple:
        return self._inputs

    @property
    def required(self) -> tuple:
        return self._required

    def call(self, **state) -> dict:
        "The named arrays after every block: the inputs and all the outputs."
        for block in self.blocks:
            args = {n: state[n] for n in block.inputs if n in state}
            missing = [n for n in block.required if n not in args]
            if missing:
                raise ValueError(f"{_name(block)} needs {missing}")
            _store(block, block(**args), state)
        return state

    def __getitem__(self, i):
        return self.blocks[i]

    def __len__(self) -> int:
        return len(self.blocks)

    def __iter__(self):
        return iter(self.blocks)

In [ ]:
class Scale(Block):
    outputs = ('x',)
    def __init__(self, factor):
        self.factor = factor
    def call(self, x, offset=0.0):
        return self.factor * x + offset

class Stats(Block):
    outputs = ('mean', 'peak')
    def call(self, x):
        return x.mean(), x.max()

pipe = Pipeline([Scale(2.0), Scale(3.0), Stats()], inputs=('x',))
out = pipe(x=np.array([1.0, 2.0]))
assert out['x'].tolist() == [6.0, 12.0] and out['mean'] == 9.0 and out['peak'] == 12.0    # x replaced in turn
assert pipe.required == ('x',) and pipe.outputs == ('x', 'mean', 'peak')
assert pipe(x=np.array([1.0]), offset=1.0)['x'].tolist() == [10.0]                          # an optional input
# the wiring is checked when the pipeline is made
try:
    Pipeline([Stats()], inputs=('y',)); raise AssertionError('no error')
except ValueError as e:
    assert "Stats needs ['x']" in str(e)
# without `inputs`, the pipeline finds them; pipelines nest
inner = Pipeline([Scale(2.0), Stats()])
assert inner.inputs == ('x', 'offset') and inner.required == ('x',)
outer = Pipeline([Scale(10.0), inner], inputs=('x',))
assert outer(x=np.array([1.0]))['peak'] == 20.0
assert [type(b).__name__ for b in outer] == ['Scale', 'Pipeline'] and len(outer) == 2

In [ ]:
# reset reaches every block, each with its own seed: the same seed, the same run
noisy = Pipeline([Draw(4), Draw(4)])
noisy.reset(seed=5); a = noisy(sender=['a'])['h']
noisy.reset(seed=5); assert (noisy(sender=['a'])['h'] == a).all()
try:
    Pipeline([lambda x: x]); raise AssertionError('no error')
except TypeError as e:
    assert 'Named' in str(e)

## Named: any callable, e.g. a Sionna block

`Named` makes a block of any callable: a function, or a Sionna block, which takes and returns tensors in order.
`inputs` names what it is given (in order, or as a dict from its keyword arguments to names), and `outputs`
names what it returns. With `numpy=True`, tensors it returns become numpy arrays, for the blocks after it.

Sionna blocks convert numpy inputs to tensors themselves (on their device, at their precision), so they mix with
comm-core blocks in a `Pipeline`.

In [ ]:
#| export
def _to_numpy(x):
    if isinstance(x, tuple):
        return tuple(_to_numpy(v) for v in x)
    if isinstance(x, list):
        return [_to_numpy(v) for v in x]
    if isinstance(x, dict):
        return {k: _to_numpy(v) for k, v in x.items()}
    if type(x).__module__.partition('.')[0] == 'torch':
        return x.detach().cpu().numpy()
    return x


class Named(Block):
    "Any callable as a block (e.g. a Sionna block): the names of its inputs and of its outputs."

    def __init__(
        self,
        block: Callable, # A callable: a function, a Sionna block, ...
        inputs: Sequence[str] | Mapping[str, str], # Names of its arguments, in order; or {keyword argument: name}
        outputs: Sequence[str] | str, # Names of what it returns, in order
        numpy: bool = False, # Return numpy arrays instead of torch tensors
        **kwargs, # Fixed keyword arguments of every call
    ):
        self.block, self.numpy, self.kwargs = block, numpy, kwargs
        self._names = dict(inputs) if isinstance(inputs, Mapping) else tuple(inputs)
        self.outputs = (outputs,) if isinstance(outputs, str) else tuple(outputs)

    @property
    def inputs(self) -> tuple:
        return tuple(self._names.values()) if isinstance(self._names, dict) else self._names

    @property
    def required(self) -> tuple:
        return self.inputs

    def call(self, **named):
        if isinstance(self._names, dict):
            out = self.block(**{k: named[n] for k, n in self._names.items()}, **self.kwargs)
        else:
            out = self.block(*(named[n] for n in self._names), **self.kwargs)
        return _to_numpy(out) if self.numpy else out

In [ ]:
# a function, by position or by keyword, with fixed arguments
power = Named(lambda h, p: p * np.abs(h) ** 2, ('h', 'power'), 'rx_power')
assert power.inputs == ('h', 'power') and power(h=np.array([2.0]), power=np.array([0.5]))[0] == 2.0
clip = Named(np.clip, {'a': 'x'}, 'x', a_min=0, a_max=1)
out = Pipeline([Scale(2.0), clip], inputs=('x',))(x=np.array([-1.0, 0.2, 3.0]))
assert out['x'].tolist() == [0.0, 0.4, 1.0]

In [ ]:
import importlib

Sionna PHY's blocks, composed with a comm-core block: 16-QAM over AWGN, the bits' error rate counted by a block
of ours.

In [ ]:
#| sionna
AWGN = importlib.import_module('sionna.phy.channel').AWGN
mapping = importlib.import_module('sionna.phy.mapping')
BinarySource, Demapper, Mapper = mapping.BinarySource, mapping.Demapper, mapping.Mapper

class BitErrors(Block):
    outputs = ('ber',)
    def call(self, bits, llr):
        return float(((llr > 0) != (bits > 0.5)).mean())

qam = Pipeline([
    Named(BinarySource(), ('shape',), 'bits', numpy=True),
    Named(Mapper('qam', 4), ('bits',), 'x'),
    Named(AWGN(), ('x', 'no'), 'y'),
    Named(Demapper('app', 'qam', 4), ('y', 'no'), 'llr', numpy=True),
    BitErrors(),
], inputs=('shape', 'no'))
out = qam(shape=[100, 400], no=10 ** (-20 / 10))            # 20 dB
noisy = qam(shape=[100, 400], no=10 ** (-10 / 10))          # 10 dB
print(f"16-QAM bit error rate: {out['ber']:.4f} at 20 dB, {noisy['ber']:.3f} at 10 dB")
assert out['ber'] < 1e-3 < noisy['ber'] and isinstance(out['llr'], np.ndarray)

## The names

The blocks of comm-core use these names, so that any of them works in any pipeline. For a step of a `Network`,
`PipelineBackend` (see `backend`) gives the inputs, and turns the outputs into the results. Each array has one
row per transmission, in the order of the step.

| Name | Shape | What |
|---|---|---|
| `transmission`, `link` | list | The transmissions of the step, and their links (inputs) |
| `sender`, `receiver` | list | Their nodes (inputs) |
| `size_bits` | (n,) | `message.size_bits`, 0 if none (input) |
| `group` | list | The broadcast of each, None for a unicast (input) |
| `noise_power` | number | Noise power at the receivers (input) |
| `h` | (n, K) | Channel coefficient at each of K subcarriers (one: flat); `inf`: the nodes at the same place |
| `power`, `active` | (n,) | Transmit power, and whether the transmission is sent and received (False: an outage) |
| `interference`, `half_duplex` | (n,) | Power of the other senders at the receiver; whether the receiver was transmitting |
| `sinr` | (n, K) | Linear SINR at each subcarrier |
| `per`, `num_symbols`, `info` | (n,), (n,), list | A PHY abstraction's packet error rate, symbols on the air, and metadata |
| `success`, `payload` | (n,), list | Whether each arrived, and what the receiver got (default: the payload as sent) |
| `error_rate`, `num_bits` | (n,) | The bits' error rate, and the bits on the air (link level) |
| `latency`, `energy` | (n,) | Time on the air, and transmit energy |

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()